In [1]:
import pandas as pd
import numpy as np
from time import perf_counter
from datasets import load_dataset
from memory_profiler import memory_usage
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import DistilBertTokenizer, DistilBertModel


In [2]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PRETRAINED_MODEL_NAME = 'distilbert-base-uncased-finetuned-sst-2-english'
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5 

print(f"Using device: {DEVICE}")

Using device: cuda


In [3]:
ds = load_dataset("higopires/RePro-categories-multilabel")

train_df = ds['train'].to_pandas()
val_df = ds['validation'].to_pandas()
test_df = ds['test'].to_pandas()

train_df

,review_text,ENTREGA,OUTROS,PRODUTO,CONDICOESDERECEBIMENTO,INADEQUADA,ANUNCIO
0,"Aparelho muito bom, confiável e com valor aqui...",0,0,1,0,0,0
1,"A história é muito boa, porém o autor ""enrolou...",0,0,1,0,0,0
2,"Entrega rápida, produto muito bom Amei. Pratic...",1,0,1,0,0,0
3,Produto otimo so falta o carregador da maquina...,0,0,1,1,0,0
4,a proteção anti queda não é boa se cair de fr...,0,0,1,0,0,0
...,...,...,...,...,...,...,...
7997,amei o produto. chegou no prazo e em perfeito ...,1,0,1,1,0,0
7998,Ótima embalagem. Produto entregue no prazo. Re...,1,0,1,1,0,0
7999,"ótimo produto, super recomendo .,Entrega bem r...",1,0,1,0,0,0
8000,"Veio tudo certinho, dentro do prazo e o produt...",1,0,1,1,0,0


In [4]:
train_df = train_df[train_df['INADEQUADA'] == 0].reset_index(drop=True)
val_df = val_df[val_df['INADEQUADA'] == 0].reset_index(drop=True)
test_df = test_df[test_df['INADEQUADA'] == 0].reset_index(drop=True)

train_df = train_df.drop(columns=['INADEQUADA'])
val_df = val_df.drop(columns=['INADEQUADA'])
test_df = test_df.drop(columns=['INADEQUADA'])

train_df = train_df.rename(columns={'review_text': 'text'})
val_df = val_df.rename(columns={'review_text': 'text'})
test_df = test_df.rename(columns={'review_text': 'text'})

train_df

,text,ENTREGA,OUTROS,PRODUTO,CONDICOESDERECEBIMENTO,ANUNCIO
0,"Aparelho muito bom, confiável e com valor aqui...",0,0,1,0,0
1,"A história é muito boa, porém o autor ""enrolou...",0,0,1,0,0
2,"Entrega rápida, produto muito bom Amei. Pratic...",1,0,1,0,0
3,Produto otimo so falta o carregador da maquina...,0,0,1,1,0
4,a proteção anti queda não é boa se cair de fr...,0,0,1,0,0
...,...,...,...,...,...,...
7669,amei o produto. chegou no prazo e em perfeito ...,1,0,1,1,0
7670,Ótima embalagem. Produto entregue no prazo. Re...,1,0,1,1,0
7671,"ótimo produto, super recomendo .,Entrega bem r...",1,0,1,0,0
7672,"Veio tudo certinho, dentro do prazo e o produt...",1,0,1,1,0


In [5]:
class MultiLabelClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        """
        Args:
            texts: List or array of text samples
            labels: 2D array of shape (num_samples, num_classes) with binary indicators (0 or 1)
            tokenizer: Pretrained tokenizer (e.g., DistilBertTokenizer)
            max_len: Maximum sequence length
        """
        self.texts = texts
        self.labels = labels  # Shape: (num_samples, num_classes)
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]  # Shape: (num_classes,)
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.float)  # Binary vector for multilabel
        }

In [6]:
class DistilBertForMultiLabelClassification(nn.Module):
    def __init__(self, num_classes):
        super(DistilBertForMultiLabelClassification, self).__init__()
        self.distilbert = DistilBertModel.from_pretrained(PRETRAINED_MODEL_NAME)
        self.pre_classifier = nn.Linear(768, 768)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(768, num_classes)  # Output logits for each class
    
    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = outputs[0][:, 0]  # CLS token
        pooled_output = self.pre_classifier(hidden_state)
        pooled_output = nn.ReLU()(pooled_output)
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits  # Return raw logits for BCEWithLogitsLoss

In [7]:
def get_metrics(y_true, y_pred):

    acc = accuracy_score(y_true, y_pred)

    precisions, recalls, f1s, supports = precision_recall_fscore_support(y_true, y_pred)

    return acc, precisions, recalls, f1s

In [8]:
def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, scheduler=None, epochs=EPOCHS):
    best_val_loss = float('inf')
    start_train = perf_counter()
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        train_preds = []
        train_true = []
        
        progress_bar = tqdm(train_dataloader, desc=f'Epoch {epoch + 1}/{epochs}', leave=False)
        for batch in progress_bar:
            optimizer.zero_grad()
            
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)  # Shape: (batch_size, num_classes)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)  # Shape: (batch_size, num_classes)
            loss = criterion(outputs, labels)  # BCEWithLogitsLoss
            train_loss += loss.item()
            
            # Compute binary predictions for each class
            preds = (torch.sigmoid(outputs) > 0.5).float().cpu().numpy()  # Shape: (batch_size, num_classes)
            train_preds.extend(preds)
            train_true.extend(labels.cpu().numpy())
            
            loss.backward()
            optimizer.step()
            progress_bar.set_postfix({'loss': loss.item()})
        
        if scheduler:
            scheduler.step()
            
        train_loss /= len(train_dataloader)
        train_true = np.array(train_true)  # Shape: (num_samples, num_classes)
        train_preds = np.array(train_preds)  # Shape: (num_samples, num_classes)
        
        train_acc, train_precisions, train_recalls, train_f1s = get_metrics(train_true, train_preds)
        
        start_val = perf_counter()
        model.eval()
        val_loss = 0
        val_preds = []
        val_true = []
        with torch.no_grad():
            for batch in tqdm(val_dataloader, desc="Validation", leave=False):
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                preds = (torch.sigmoid(outputs) > 0.5).float().cpu().numpy()
                val_preds.extend(preds)
                val_true.extend(labels.cpu().numpy())
        
        val_time = perf_counter() - start_val
        
        val_loss /= len(val_dataloader)
        val_true = np.array(val_true)  # Shape: (num_samples, num_classes)
        val_preds = np.array(val_preds)  # Shape: (num_samples, num_classes)
        
        val_acc, val_precisions, val_recalls, val_f1s = get_metrics(val_true, val_preds)
        
        print(f"Epoch {epoch + 1}/{epochs} - Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1s}, Prec: {train_precisions}, Recall: {train_recalls}")
        print(f"Epoch {epoch + 1}/{epochs} - Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1s}, Prec: {val_precisions}, Recall: {val_recalls}, Val Time: {val_time:.2f} sec")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'results/bert_multilabel1.pt')
            print("Model saved!")
    
    total_train_time = perf_counter() - start_train
    print(f"Total Training Time: {total_train_time:.2f} seconds")
    
    return train_acc, train_precisions, train_recalls, train_f1s, val_acc, val_precisions, val_recalls, val_f1s, total_train_time, val_time

In [9]:
def evaluate_model(model, test_dataloader):
    model.eval()
    predictions = []
    true_labels = []
    classification_times = []
    
    start_test = perf_counter()
    
    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Testing"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)  # Shape: (batch_size, num_classes)
            
            for i in range(input_ids.size(0)):
                input_id = input_ids[i].unsqueeze(0)
                attention_mask_sample = attention_mask[i].unsqueeze(0)
                label = labels[i].cpu().numpy()  # Shape: (num_classes,)
                
                start_time = perf_counter()
                
                output = model(input_ids=input_id, attention_mask=attention_mask_sample)  # Shape: (1, num_classes)
                pred = (torch.sigmoid(output) > 0.5).float().cpu().numpy()[0]  # Shape: (num_classes,)
                
                predictions.append(pred)
                true_labels.append(label)
                classification_times.append(perf_counter() - start_time)
    
    total_test_time = perf_counter() - start_test
    print(f"Test Time: {total_test_time:.2f} seconds")
    
    predictions = np.array(predictions)  # Shape: (num_samples, num_classes)
    true_labels = np.array(true_labels)  # Shape: (num_samples, num_classes)
    
    acc, precisions, recalls, f1s = get_metrics(true_labels, predictions)
    
    print("Test Metrics:")
    print("Accuracy:", acc)
    print("F1s:", f1s)
    print("Precisions:", precisions)
    print("Recalls:", recalls)
    
    return predictions, true_labels

In [10]:
train_texts = train_df['text'].values
train_labels = train_df.drop(columns=['text']).values

val_texts = val_df['text'].values
val_labels = val_df.drop(columns=['text']).values

test_texts = test_df['text'].values
test_labels = test_df.drop(columns=['text']).values

tokenizer = DistilBertTokenizer.from_pretrained(PRETRAINED_MODEL_NAME)

# Use MultiClassClassificationDataset instead of BinaryClassificationDataset
train_dataset = MultiLabelClassificationDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = MultiLabelClassificationDataset(val_texts, val_labels, tokenizer, MAX_LEN)
test_dataset = MultiLabelClassificationDataset(test_texts, test_labels, tokenizer, MAX_LEN)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

seeds = [2,3,5]

num_classes = train_labels.shape[1]

avg_train_acc = 0
avg_train_precs = np.zeros(num_classes)
avg_train_recalls = np.zeros(num_classes)
avg_train_f1s = np.zeros(num_classes)
avg_max_memory_usage_train = 0
avg_max_vram_usage_train = 0
avg_total_train_time = 0

avg_val_acc = 0
avg_val_precs = np.zeros(num_classes)
avg_val_recalls = np.zeros(num_classes)
avg_val_f1s = np.zeros(num_classes)
avg_total_val_time = 0

avg_test_acc = 0
avg_test_precs = np.zeros(num_classes)
avg_test_recalls = np.zeros(num_classes)
avg_test_f1s = np.zeros(num_classes)
avg_max_memory_usage_test = 0
avg_max_vram_usage_test = 0
avg_total_test_time = 0

for seed in seeds:
    torch.manual_seed(seed)
    model = DistilBertForMultiLabelClassification(num_classes)
    model = model.to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.BCEWithLogitsLoss()  # For multi-label

    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    max_memory_usage_train, retval = memory_usage(
        (train_model, (model, train_dataloader, val_dataloader, optimizer, criterion), {'epochs': EPOCHS}),
        max_usage=True,
        retval=True
    )

    max_vram_usage_train = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0

    (train_acc, train_precisions, train_recalls, train_f1s,
     val_acc, val_precisions, val_recalls, val_f1s,
     total_train_time, val_time) = retval

    model.load_state_dict(torch.load('results/bert_multilabel1.pt'))

    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start = perf_counter()
    max_memory_usage_test, retval = memory_usage(
        (evaluate_model, (model, test_dataloader), {}),
        max_usage=True,
        retval=True
    )
    total_time_test = perf_counter() - start

    max_vram_usage_test = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0

    predictions, true_labels = retval

    test_acc, test_precisions, test_recalls, test_f1s = get_metrics(true_labels, predictions)

    avg_train_acc += train_acc
    avg_train_precs += train_precisions
    avg_train_recalls += train_recalls
    avg_train_f1s += train_f1s
    avg_max_memory_usage_train += max_memory_usage_train
    avg_max_vram_usage_train += max_vram_usage_train
    avg_total_train_time += total_train_time

    avg_val_acc += val_acc
    avg_val_precs += val_precisions
    avg_val_recalls += val_recalls
    avg_val_f1s += val_f1s
    avg_total_val_time += val_time

    avg_test_acc += test_acc
    avg_test_precs += test_precisions
    avg_test_recalls += test_recalls
    avg_test_f1s += test_f1s
    avg_max_memory_usage_test += max_memory_usage_test
    avg_max_vram_usage_test += max_vram_usage_test
    avg_total_test_time += total_time_test

avg_train_acc /= len(seeds)
avg_train_precs /= len(seeds)
avg_train_recalls /= len(seeds)
avg_train_f1s /= len(seeds)
avg_max_memory_usage_train /= len(seeds)
avg_max_vram_usage_train /= len(seeds)
avg_total_train_time /= len(seeds)

avg_val_acc /= len(seeds)
avg_val_precs /= len(seeds)
avg_val_recalls /= len(seeds)
avg_val_f1s /= len(seeds)
avg_total_val_time /= len(seeds)

avg_test_acc /= len(seeds)
avg_test_precs /= len(seeds)
avg_test_recalls /= len(seeds)
avg_test_f1s /= len(seeds)
avg_max_memory_usage_test /= len(seeds)
avg_max_vram_usage_test /= len(seeds)
avg_total_test_time /= len(seeds)

avg_classification_time = avg_total_test_time / len(test_texts)

avg_classification_time

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1/3 - Train Loss: 0.3511, Acc: 0.5774, F1: [0.75810353 0.50744102 0.90816963 0.38348416 0.00867052], Prec: [0.89594054 0.68934911 0.86488493 0.7739726  0.16666667], Recall: [0.65702306 0.40149339 0.9560151  0.25488722 0.00445104]
Epoch 1/3 - Val Loss: 0.2378, Acc: 0.6744, F1: [0.91965812 0.71794872 0.94729908 0.60082305 0.        ], Prec: [0.93728223 0.8045977  0.9510582  0.85882353 0.        ], Recall: [0.90268456 0.64814815 0.94356955 0.46202532 0.        ], Val Time: 3.82 sec
Model saved!


Epoch 2/3 - Train Loss: 0.2105, Acc: 0.7097, F1: [0.92500531 0.76658625 0.93874563 0.71928328 0.54954035], Prec: [0.93755383 0.80698413 0.93064516 0.83136095 0.88196721], Recall: [0.91278826 0.73004021 0.94698835 0.63383459 0.39910979]
Epoch 2/3 - Val Loss: 0.1727, Acc: 0.7458, F1: [0.94117647 0.79058824 0.95069532 0.74368231 0.81632653], Prec: [0.97142857 0.80382775 0.91707317 0.86554622 0.9375    ], Recall: [0.91275168 0.77777778 0.98687664 0.65189873 0.72289157], Val Time: 3.75 sec
Model saved!


Epoch 3/3 - Train Loss: 0.1547, Acc: 0.7726, F1: [0.94874499 0.8368669  0.95065359 0.79935406 0.78828452], Prec: [0.95458404 0.85861027 0.94647796 0.86312119 0.90403071], Recall: [0.94297694 0.81619759 0.95486624 0.7443609  0.69881306]
Epoch 3/3 - Val Loss: 0.1602, Acc: 0.7637, F1: [0.94983278 0.8091954  0.94722779 0.78378378 0.84768212], Prec: [0.94666667 0.80365297 0.96462585 0.84057971 0.94117647], Recall: [0.95302013 0.81481481 0.93044619 0.73417722 0.77108434], Val Time: 3.77 sec
Model saved!
Total Training Time: 255.55 seconds


C:\Users\Rafael\AppData\Local\Temp\ipykernel_20292\2316905420.py:70: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('results/bert_multilabel1

...

Epoch 2/3 - Train Loss: 0.2205, Acc: 0.6998, F1: [0.91627709 0.74191542 0.93515745 0.69100391 0.5960396 ], Prec: [0.92890995 0.80881356 0.92268371 0.81874356 0.89583333], Recall: [0.90398323 0.68523837 0.94797308 0.59774436 0.44658754]
Epoch 2/3 - Val Loss: 0.1818, Acc: 0.7300, F1: [0.93377483 0.75252525 0.94257426 0.72924188 0.77464789], Prec: [0.92156863 0.82777778 0.94820717 0.8487395  0.93220339], Recall: [0.94630872 0.68981481 0.93700787 0.63924051 0.6626506 ], Val Time: 3.70 sec
Model saved!


Epoch 3/3 - Train Loss: 0.1635, Acc: 0.7652, F1: [0.9468333  0.81861292 0.94851743 0.78947368 0.76435304], Prec: [0.9567637  0.84568279 0.9416141  0.85526316 0.90466531], Recall: [0.93710692 0.79322229 0.95552273 0.73308271 0.66172107]
Epoch 3/3 - Val Loss: 0.1619, Acc: 0.7826, F1: [0.94370861 0.83295195 0.95543906 0.79591837 0.8028169 ], Prec: [0.93137255 0.82352941 0.95418848 0.86029412 0.96610169], Recall: [0.95637584 0.84259259 0.95669291 0.74050633 0.68674699], Val Time: 3.71 sec
Model saved!
Total Training Time: 262.13 seconds


C:\Users\Rafael\AppData\Local\Temp\ipykernel_20292\2316905420.py:70: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('results/bert_multilabel1

Test Time: 6.35 seconds
Test Metrics:
Accuracy: 0.7556935817805382
F1s: [0.93924466 0.80266075 0.9486166  0.80645161 0.75342466]
Precisions: [0.91961415 0.78017241 0.95112285 0.85616438 0.88709677]
Recalls: [0.95973154 0.82648402 0.94612352 0.76219512 0.6547619 ]


0.007128251414780834

In [11]:
# save results to txt
with open("results/bert_multilabel1.txt", "w") as f:
    f.write(f"Average Train Accuracy: {avg_train_acc}\n")
    f.write(f"Average Train Precisions: {avg_train_precs}\n")
    f.write(f"Average Train Recalls: {avg_train_recalls}\n")
    f.write(f"Average Train F1s: {avg_train_f1s}\n")
    f.write(f"Average Max Memory Usage Train: {avg_max_memory_usage_train}\n")
    f.write(f"Average Max VRAM Usage Train: {avg_max_vram_usage_train}\n")
    f.write(f"Average Total Train Time: {avg_total_train_time}\n")
    f.write("\n")
    f.write(f"Average Val Accuracy: {avg_val_acc}\n")
    f.write(f"Average Val Precisions: {avg_val_precs}\n")
    f.write(f"Average Val Recalls: {avg_val_recalls}\n")
    f.write(f"Average Val F1s: {avg_val_f1s}\n")
    f.write(f"Average Total Val Time: {avg_total_val_time}\n")
    f.write("\n")
    f.write(f"Average Test Accuracy: {avg_test_acc}\n")
    f.write(f"Average Test Precisions: {avg_test_precs}\n")
    f.write(f"Average Test Recalls: {avg_test_recalls}\n")
    f.write(f"Average Test F1s: {avg_test_f1s}\n")
    f.write(f"Average Max Memory Usage Test: {avg_max_memory_usage_test}\n")
    f.write(f"Average Max VRAM Usage Test: {avg_max_vram_usage_test}\n")
    f.write(f"Average Total Test Time: {avg_total_test_time}\n")
    f.write("\n")
    f.write(f"Average Classification Time: {avg_classification_time}\n")
    f.write(f"Lines classified {len(test_texts)}\n")

    f.close()